In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    print("Завантаження датасету (це може зайняти хвилину-дві, файл близько 20 МБ)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Розпакування...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Успішно розпаковано!")
else:
    print("Файл вже існує, пропускаємо завантаження.")

print("Зчитування даних (може зайняти кілька секунд)...")
df = pd.read_csv(txt_path, sep=';', na_values='?', low_memory=False)

original_shape = df.shape
df = df.dropna()
cleaned_shape = df.shape

print(f"Очищення завершено! Було рядків: {original_shape[0]}, залишилось: {cleaned_shape[0]}")

df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

display(df.head())

Завантаження датасету (це може зайняти хвилину-дві, файл близько 20 МБ)...
Розпакування...
Успішно розпаковано!
Зчитування даних (може зайняти кілька секунд)...
Очищення завершено! Було рядків: 2075259, залишилось: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [2]:
import timeit

def filter_high_power(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

start_time = timeit.default_timer()
high_power_df = filter_high_power(df)
execution_time = timeit.default_timer() - start_time

print(f"Знайдено записів: {len(high_power_df)}")
print(f"Час виконання: {execution_time:.5f} секунд")
display(high_power_df.head())

Знайдено записів: 17547
Час виконання: 0.05112 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


In [3]:
def filter_intensity_and_appliances(dataframe):
    filtered_df = dataframe[(dataframe['Global_intensity'] >= 19.0) & (dataframe['Global_intensity'] <= 20.0)]
    
    result_df = filtered_df[filtered_df['Sub_metering_2'] > filtered_df['Sub_metering_3']]
    return result_df

start_time = timeit.default_timer()
intensity_df = filter_intensity_and_appliances(df)
execution_time = timeit.default_timer() - start_time

print(f"Знайдено записів: {len(intensity_df)}")
print(f"Час виконання: {execution_time:.5f} секунд")
display(intensity_df.head())

Знайдено записів: 2509
Час виконання: 0.05186 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


In [7]:
def random_sample_means(dataframe):
    sampled_df = dataframe.sample(n=500000, replace=False, random_state=42)
    means = {
        'Sub_metering_1_mean': sampled_df['Sub_metering_1'].mean(),
        'Sub_metering_2_mean': sampled_df['Sub_metering_2'].mean(),
        'Sub_metering_3_mean': sampled_df['Sub_metering_3'].mean()
    }
    return means

start_time = timeit.default_timer()
sample_means = random_sample_means(df)
execution_time = timeit.default_timer() - start_time

print(f"Час виконання: {execution_time:.5f} секунд")
print("Середні величини споживання для 500 000 записів:")
for k, v in sample_means.items():
    print(f"{k}: {v:.4f}")

Час виконання: 0.14566 секунд
Середні величини споживання для 500 000 записів:
Sub_metering_1_mean: 1.1193
Sub_metering_2_mean: 1.3089
Sub_metering_3_mean: 6.4530


In [6]:
def complex_evening_filter(dataframe):
    evening_df = dataframe[dataframe['Time'] >= '18:00:00']
    
    high_power_evening = evening_df[evening_df['Global_active_power'] > 6.0]
    
    group2_dominant = high_power_evening[
        (high_power_evening['Sub_metering_2'] > high_power_evening['Sub_metering_1']) & 
        (high_power_evening['Sub_metering_2'] > high_power_evening['Sub_metering_3'])
    ]
    
    group2_dominant = group2_dominant.reset_index(drop=True)
    half_index = len(group2_dominant) // 2
    
    first_half = group2_dominant.iloc[:half_index:3]
    second_half = group2_dominant.iloc[half_index::4]
    
    final_result = pd.concat([first_half, second_half])
    return final_result

start_time = timeit.default_timer()
complex_df = complex_evening_filter(df)
execution_time = timeit.default_timer() - start_time

print(f"Знайдено записів після всіх фільтрів: {len(complex_df)}")
print(f"Час виконання: {execution_time:.5f} секунд")
display(complex_df.head())

Знайдено записів після всіх фільтрів: 310
Час виконання: 0.16618 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


In [17]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

cols_to_scale = ['Global_active_power', 'Voltage']
data_to_scale = df[cols_to_scale].dropna() # беремо без пропусків про всяк випадок

scaler_standard = StandardScaler()
standardized_data = scaler_standard.fit_transform(data_to_scale)

scaler_minmax = MinMaxScaler()
normalized_data = scaler_minmax.fit_transform(data_to_scale)

standard_df = pd.DataFrame(standardized_data, columns=[f"{c}_standardized" for c in cols_to_scale])
normal_df = pd.DataFrame(normalized_data, columns=[f"{c}_normalized" for c in cols_to_scale])

print("Стандартизовані дані (перші 5 записів):")
display(standard_df.head())

print("\nПронормовані дані (перші 5 записів):")
display(normal_df.head())

Стандартизовані дані (перші 5 записів):


,Global_active_power_standardized,Voltage_standardized
0,2.955077,-1.851816
1,4.037085,-2.225274
2,4.050326,-2.330213
3,4.063567,-2.191324
4,2.434881,-1.592556



Пронормовані дані (перші 5 записів):


,Global_active_power_normalized,Voltage_normalized
0,0.374796,0.376090
1,0.478363,0.336995
2,0.479631,0.326010
3,0.480898,0.340549
4,0.325005,0.403231


In [18]:
pearson_corr = df['Global_active_power'].corr(df['Voltage'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Voltage'], method='spearman')

print(f"Кореляція Пірсона (Global_active_power vs Voltage): {pearson_corr:.4f}")
print(f"Кореляція Спірмена (Global_active_power vs Voltage): {spearman_corr:.4f}")

df['Day_of_week'] = df['Datetime'].dt.day_name()

ohe_df = pd.get_dummies(df['Day_of_week'].head(10000), prefix='Day')

print("\nРезультат One Hot Encoding (перші 5 записів для колонки 'День тижня'):")
display(ohe_df.head())

Кореляція Пірсона (Global_active_power vs Voltage): -0.3998
Кореляція Спірмена (Global_active_power vs Voltage): -0.3252

Результат One Hot Encoding (перші 5 записів для колонки 'День тижня'):


,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
0,False,False,True,False,False,False,False
1,False,False,True,False,False,False,False
2,False,False,True,False,False,False,False
3,False,False,True,False,False,False,False
4,False,False,True,False,False,False,False
